In [1]:
import numpy as np

from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from utils import load_preprocessed_ds

In [2]:
X_tr, y_tr, X_te, y_te, X_tr_pd = load_preprocessed_ds()

Gini impurity is,
if you take a random sample from a dataset,
and assign it randomly to a class,
what is the probability this class is wrong?

In [3]:
def gini(y): # y is labels
    labels_count = len(y)
    if labels_count == 0:
        return 0
    _, frequencies = np.unique(y, return_counts=True)
    probabilities = np.sum((frequencies / labels_count) ** 2)
    return 1 - probabilities

# gini(y_train)

In [4]:
def split_labels(y, feature, threshold):
    y_left = y[feature <= threshold]
    y_right = y[feature > threshold]
    return y_left, y_right

def weighted_gini(y_left, y_right):
    len_left = len(y_left)
    len_right = len(y_right)
    len_total = len_left + len_right
    return (len_left * gini(y_left) + len_right * gini(y_right)) / len_total

# Test
# age_idx = X_train.columns.get_loc("age")
#
# y_left, y_right = split_labels(y_tr, X_tr[:, age_idx], 0.1)
#
# weighted_gini(y_left, y_right)

In [5]:
def find_best_split(X, y):
    lowest_gini = np.inf
    feature_at_lowest_gini = None
    thresh_at_lowest_gini = None

    for feature_idx in range(X.shape[1]):

        unique_sorted_vals = np.unique(X[:, feature_idx])
        thresholds = (unique_sorted_vals[:-1] + unique_sorted_vals[1:]) / 2

        for threshold in thresholds:
            y_left, y_right = split_labels(y, X[:,feature_idx], threshold)
            current_gini = weighted_gini(y_left, y_right)

            if current_gini < lowest_gini:
                lowest_gini = current_gini
                feature_at_lowest_gini = feature_idx
                thresh_at_lowest_gini = threshold
    return lowest_gini, feature_at_lowest_gini, thresh_at_lowest_gini

find_best_split(X_tr, y_tr)

(np.float64(0.3642147663558955), 15, np.float64(0.5))

In [6]:
# Tree constants
MIN_SAMPLES_SPLIT = 10
MAX_DEPTH = 3

In [7]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.is_leaf = self.value is not None

    def predict_one(self, x):
        if self.is_leaf:
            return self.value
        else:
            if x[self.feature] <= self.threshold:
                return self.left.predict_one(x)
            else:
                return self.right.predict_one(x)

    def predict_batch(self, X):
        return np.array([self.predict_one(X[i,:]) for i in range(X.shape[0])])


def most_common_label(y):
    vals, counts = np.unique(y, return_counts=True)
    return vals[np.argmax(counts)]


def build_tree(X, y, depth=0, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT):

    # stopping conditions
    reached_max_depth = depth == max_depth
    too_little_samples_left = len(y) < min_samples_split
    all_samples_same_class = len(np.unique(y)) == 1
    if reached_max_depth or too_little_samples_left or all_samples_same_class:
        return Node(value=most_common_label(y)) # leaf

    lowest_weighted_gini, feature, thresh = find_best_split(X, y)

    # another stopping condition
    no_improvement_in_splitting = lowest_weighted_gini >= gini(y)
    if no_improvement_in_splitting:
        return Node(value=most_common_label(y)) # leaf

    # split X and y
    mask = X[:, feature] <= thresh
    X_left = X[mask, :]
    y_left = y[mask]
    X_right = X[~mask, :]
    y_right = y[~mask]

    # recurse
    left_node = build_tree(X_left, y_left, depth + 1, max_depth, min_samples_split)
    right_node = build_tree(X_right, y_right, depth + 1, max_depth, min_samples_split)

    return Node(feature=feature, threshold=thresh, left=left_node, right=right_node)

In [8]:
root = build_tree(X_tr, y_tr, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT)
preds = root.predict_batch(X_te)
accuracy = np.mean(preds == y_te)
print(f"accuracy: {accuracy}")

accuracy: 0.7213114754098361


In [9]:
# Sanity check with sklearn
clf = DecisionTreeClassifier(criterion='gini', max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT, random_state=1)
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_te)
accuracy = accuracy_score(y_te, y_pred)
print(f'Accuracy: {accuracy}')

Accuracy: 0.7213114754098361


In [10]:
def print_tree_sklearn_style(node, feature_names, depth=0):
    indent = "|   " * depth
    if node.is_leaf:
        print(f"{indent}|--- class: {node.value}")
    else:
        fname = feature_names[node.feature]
        print(f"{indent}|--- {fname} <= {node.threshold:.2f}")
        print_tree_sklearn_style(node.left, feature_names, depth + 1)
        print(f"{indent}|--- {fname} >  {node.threshold:.2f}")
        print_tree_sklearn_style(node.right, feature_names, depth + 1)

print_tree_sklearn_style(root, list(X_tr_pd.columns))

|--- thal_2.0 <= 0.50
|   |--- oldpeak <= -0.42
|   |   |--- ca_1.0 <= 0.50
|   |   |   |--- class: 1.0
|   |   |--- ca_1.0 >  0.50
|   |   |   |--- class: 0.0
|   |--- oldpeak >  -0.42
|   |   |--- thalach <= -0.23
|   |   |   |--- class: 0.0
|   |   |--- thalach >  -0.23
|   |   |   |--- class: 0.0
|--- thal_2.0 >  0.50
|   |--- oldpeak <= 0.99
|   |   |--- exang <= 0.50
|   |   |   |--- class: 1.0
|   |   |--- exang >  0.50
|   |   |   |--- class: 1.0
|   |--- oldpeak >  0.99
|   |   |--- class: 0.0


In [11]:
from sklearn.tree import export_text
print(export_text(clf, feature_names=list(X_tr_pd.columns)))

|--- thal_2.0 <= 0.50
|   |--- oldpeak <= -0.42
|   |   |--- ca_1.0 <= 0.50
|   |   |   |--- class: 1.0
|   |   |--- ca_1.0 >  0.50
|   |   |   |--- class: 0.0
|   |--- oldpeak >  -0.42
|   |   |--- thalach <= -0.23
|   |   |   |--- class: 0.0
|   |   |--- thalach >  -0.23
|   |   |   |--- class: 0.0
|--- thal_2.0 >  0.50
|   |--- oldpeak <= 0.99
|   |   |--- exang <= 0.50
|   |   |   |--- class: 1.0
|   |   |--- exang >  0.50
|   |   |   |--- class: 1.0
|   |--- oldpeak >  0.99
|   |   |--- class: 0.0

